In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('model_ready.csv')

# Basic inspection
print(df.info())
print(df.head())
print(df.describe())
print("Target counts:\n", df['label_3class'].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 11945 entries, 0 to 11944
Data columns (total 23 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   source_folder                11945 non-null  str    
 1   source_session_path          11945 non-null  str    
 2   session_id                   11945 non-null  str    
 3   participant_id               11945 non-null  str    
 4   timestamp_ms                 11945 non-null  int64  
 5   label_3class                 11945 non-null  str    
 6   delta_pct                    11945 non-null  float64
 7   theta_pct                    11945 non-null  float64
 8   low_alpha_pct                11945 non-null  float64
 9   high_alpha_pct               11945 non-null  float64
 10  low_beta_pct                 11945 non-null  float64
 11  high_beta_pct                11945 non-null  float64
 12  low_gamma_pct                11945 non-null  float64
 13  mid_gamma_pct              

In [ ]:
import numpy as np

# 1. Time since session start
df['session_time_sec'] = df.groupby('session_id')['timestamp_ms'].transform(lambda x: (x - x.min()) / 1000.0)

# 2. EEG Band Ratios (Adding a small epsilon to denominator to avoid division by zero)
eps = 1e-6
df['theta_beta_ratio'] = df['theta_pct'] / (df['low_beta_pct'] + df['high_beta_pct'] + eps)
df['alpha_beta_ratio'] = (df['low_alpha_pct'] + df['high_alpha_pct']) / (df['low_beta_pct'] + df['high_beta_pct'] + eps)
df['slow_fast_ratio'] = (df['delta_pct'] + df['theta_pct']) / (df['low_beta_pct'] + df['high_beta_pct'] + df['low_gamma_pct'] + df['mid_gamma_pct'] + eps)

# 3. Rolling Statistics (Mean and Std) for key features
# We group by session_id to ensure we don't bleed data across sessions
eeg_cols = ['delta_pct', 'theta_pct', 'low_alpha_pct', 'high_alpha_pct', 'low_beta_pct', 'high_beta_pct', 'attention', 'meditation']
window_size = 5

for col in eeg_cols:
    df[f'{col}_roll_mean_{window_size}'] = df.groupby('session_id')[col].transform(lambda x: x.rolling(window=window_size, min_periods=1).mean())
    df[f'{col}_roll_std_{window_size}'] = df.groupby('session_id')[col].transform(lambda x: x.rolling(window=window_size, min_periods=1).std().fillna(0))

# 4. Target Encoding
label_map = {'calm': 0, 'neutral': 1, 'stressed': 2}
df['label_encoded'] = df['label_3class'].map(label_map)

# 5. Drop redundant columns if any
# source_folder and source_session_path might not be useful for the model
cols_to_drop = ['source_folder', 'source_session_path']
df_final = df.drop(columns=cols_to_drop)

# Check the new shape and columns
print(df_final.info())
print(df_final.head())

# Save to CSV
df_final.to_csv('featured_mindtune.csv', index=False)